# AI Tools Claude by Anthropic

## Claude Model Family

| Model | Speed | Intelligence | Best For |
|-------|-------|-------------|----------|
| **Claude Haiku 4.5** | Fastest | Good | High-volume, simple tasks |
| **Claude Sonnet 4.6** | Balanced | Great | Most tasks, coding, analysis |
| **Claude Opus 4.8** | Slower | Best | Complex reasoning, research |

Context window: up to **200K tokens** (Sonnet/Opus).

---

## Effective Prompting for Claude

### XML Tag Formatting (Claude's Strong Preference)
Claude responds especially well to structured XML:
```xml
<task>Analyze the following code</task>
<code language="python">
def factorial(n):
    return n * factorial(n-1)
</code>
<requirements>
  <req>Identify bugs</req>
  <req>Suggest improvements</req>
</requirements>
```

### Extended Thinking
Claude can use a hidden scratchpad for complex reasoning:
- Enabled with `thinking: {type: 'enabled', budget_tokens: N}`
- Improves math, coding, and multi-step reasoning
- Thinking tokens are NOT billed at the same rate as output tokens

---

## Tool Use / Function Calling

Claude supports structured tool use:
1. You define tools with JSON schemas
2. Claude decides when to call them
3. You execute the tool and return results
4. Claude uses results in its response

---

## Prompt Caching

Cache large, static prompt prefixes to reduce latency and cost:
- Mark with `cache_control: {type: 'ephemeral'}`
- Cache TTL: 5 minutes (refreshed on use)
- Up to **90% cost reduction** for cached tokens
- Minimum cacheable block: 1024 tokens (Sonnet/Opus)

---

## Vision Capabilities

Claude accepts images in:
- Base64-encoded format
- URL references
- PDF documents (reads natively)

Supported: JPEG, PNG, GIF, WebP. Max 5MB per image.

In [1]:
# pip install anthropic
import os
import anthropic

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# ── Basic Message ─────────────────────────────────────────────────────────────
msg = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=512,
    system="You are an expert Python developer. Be concise.",
    messages=[{
        "role": "user",
        "content": "Explain list comprehensions with 2 examples."
    }]
)
print(msg.content[0].text)

In [2]:
# ── XML Tag Structured Prompt ─────────────────────────────────────────────────
msg = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    messages=[{
        "role": "user",
        "content": """
        <task>Review this Python function and provide feedback</task>
        <code>
        def get_user(id):
            users = load_all_users()  # loads 1M users
            for user in users:
                if user['id'] == id:
                    return user
        </code>
        <focus>
          <item>Performance issues</item>
          <item>Best practices</item>
          <item>Improved version</item>
        </focus>
        """
    }]
)
print(msg.content[0].text)

In [3]:
# ── Tool Use ──────────────────────────────────────────────────────────────────
import json

tools = [{
    "name": "get_weather",
    "description": "Get current weather for a location",
    "input_schema": {
        "type": "object",
        "properties": {
            "location": {"type": "string", "description": "City and country"},
            "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
        },
        "required": ["location"]
    }
}]

def get_weather(location: str, unit: str = "celsius") -> dict:
    # Simulated weather API
    return {"location": location, "temperature": 22, "unit": unit, "condition": "sunny"}

messages = [{"role": "user", "content": "What's the weather in Paris?"}]

# First call Claude may request a tool
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=512,
    tools=tools,
    messages=messages
)

# Process tool calls
if response.stop_reason == "tool_use":
    for block in response.content:
        if block.type == "tool_use":
            tool_result = get_weather(**block.input)
            messages.append({"role": "assistant", "content": response.content})
            messages.append({
                "role": "user",
                "content": [{"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(tool_result)}]
            })
    
    final = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=512,
        tools=tools,
        messages=messages
    )
    print(final.content[0].text)

In [4]:
# ── Streaming ─────────────────────────────────────────────────────────────────
with client.messages.stream(
    model="claude-sonnet-4-6",
    max_tokens=256,
    messages=[{"role": "user", "content": "Count from 1 to 10 slowly."}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
print()

In [5]:
# ── Prompt Caching ────────────────────────────────────────────────────────────
large_document = "A" * 2000  # Simulating a large document

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=256,
    system=[{
        "type": "text",
        "text": f"You are analyzing this document: {large_document}",
        "cache_control": {"type": "ephemeral"}  # Cache this large prefix
    }],
    messages=[{"role": "user", "content": "Summarize the document."}]
)

usage = response.usage
print(f"Input tokens: {usage.input_tokens}")
print(f"Cache created: {getattr(usage, 'cache_creation_input_tokens', 0)}")
print(f"Cache read: {getattr(usage, 'cache_read_input_tokens', 0)}")

## Additional Learning Resources

### Documentation
- [Anthropic API Docs](https://docs.anthropic.com/en/api/getting-started)
- [Claude Prompt Library](https://docs.anthropic.com/en/prompt-library/library)
- [Tool Use Guide](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)
- [Prompt Caching Guide](https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching)
- [Extended Thinking](https://docs.anthropic.com/en/docs/build-with-claude/extended-thinking)

### Cookbooks
- [Anthropic Cookbook (GitHub)](https://github.com/anthropics/anthropic-cookbook)
- [Claude Code Documentation](https://docs.anthropic.com/en/docs/claude-code/overview)

### Research
- [Constitutional AI Paper](https://arxiv.org/abs/2212.08073)
- [Model Cards](https://www.anthropic.com/model-card)